In [93]:
!pip install pandas numpy gdown

In [94]:
import pandas as pd
import gdown

file_id = "19hOT_T7Jr82N6u9XXSBEcM7P2GI-6rWL"
url = f"https://drive.google.com/uc?id={file_id}"

output = "sample.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df.head()

Downloading...
From: https://drive.google.com/uc?id=19hOT_T7Jr82N6u9XXSBEcM7P2GI-6rWL
To: /content/sample.csv
100%|██████████| 4.40M/4.40M [00:00<00:00, 174MB/s]


,title,text,category_id
0,Івано-Франківський драмтеатр знайшов архівні д...,{'clean': 'Про це на пресконференції повідомив...,0
1,Премія Олеся Гончара оголосила номінантів,"{'clean': 'Як передає Укрінформ, про це повідо...",0
2,Біографічний фільм «Я граю Роккі» вийде у лист...,"{'clean': 'Як передає Укрінформ, про це повідо...",0
3,Netflix підписав угоду на трансляцію фільмів S...,"{'clean': 'Про це повідомила Deadline, передає...",0
4,На Венеційській бієнале Україну представить ск...,{'clean': 'Про це розповіли учасники Венеційсь...,0


In [95]:
from ast import literal_eval

texts = []

for i, row in df.iterrows():
    text_dict = literal_eval(row["text"])
    clean_text = text_dict["clean"]

    texts.append({
        "text_id": i,
        "text": clean_text
    })

texts_df = pd.DataFrame(texts)
texts_df.head()

,text_id,text
0,0,Про це на пресконференції повідомив гендиректо...
1,1,"Як передає Укрінформ, про це повідомило Держми..."
2,2,"Як передає Укрінформ, про це повідомляє The Ho..."
3,3,"Про це повідомила Deadline, передає Укрінформ...."
4,4,Про це розповіли учасники Венеційської бієнале...


In [96]:
import re

months = r"(січня|лютого|березня|квітня|травня|червня|липня|серпня|вересня|жовтня|листопада|грудня)"
DATE_PATTERN = re.compile(rf"\b\d{{1,2}}(?:\s+та\s+\d{{1,2}})?\s+{months}(?:\s+\d{{4}})?", re.IGNORECASE)
AMOUNT_PATTERN = re.compile(r"\b\d+[.,]?\d*\s?(млн|млрд)?\s?(грн|₴|€|uah|долар(ів)?|usd|eur)\b", re.IGNORECASE)
SCORE_PATTERN = re.compile(r"\b\d{1,2}[:\-–]\d{1,2}\b")


def extract_dates(text_id, text):

    results = []

    for m in DATE_PATTERN.finditer(text):

        results.append({
            "text_id": text_id,
            "field_type": "DATE",
            "value": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_date_v1"
        })

    return results


def extract_amounts(text_id, text):

    results = []

    for m in AMOUNT_PATTERN.finditer(text):

        results.append({
            "text_id": text_id,
            "field_type": "AMOUNT",
            "value": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_amount_v1"
        })

    return results


def extract_scores(text_id, text):

    results = []

    for m in SCORE_PATTERN.finditer(text):

        results.append({
            "text_id": text_id,
            "field_type": "SCORE",
            "value": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_score_v1"
        })

    return results


def extract_all(text_id, text):

    return {
        "DATE": extract_dates(text_id, text),
        "AMOUNT": extract_amounts(text_id, text),
        "SCORE": extract_scores(text_id, text)
    }

In [97]:
import json
all_results = []

for _, row in texts_df.iterrows():

    extracted = extract_all(row["text_id"], row["text"])

    for field_type in extracted:
        all_results.extend(extracted[field_type])

with open("ie_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

In [98]:
import json
file_id = "1nrgN0bB65_mSLTgPY9qb4moJvoIxezmK"
url = f"https://drive.google.com/uc?id={file_id}"

output_path = "lab4_gold_ie.jsonl"
gdown.download(url, output_path, quiet=False)
cases = []
with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        cases.append(json.loads(line))

gold = pd.read_json("lab4_gold_ie.jsonl", lines=True)
pred = pd.DataFrame(all_results)

Downloading...
From: https://drive.google.com/uc?id=1nrgN0bB65_mSLTgPY9qb4moJvoIxezmK
To: /content/lab4_gold_ie.jsonl
100%|██████████| 267k/267k [00:00<00:00, 51.0MB/s]


In [99]:
pred = pred.rename(columns={"value": "span_text"})

In [100]:
gold_text_ids = gold["text_id"].unique()
pred_filtered = pred[pred["text_id"].isin(gold_text_ids)]

merged = pred_filtered.merge(
    gold,
    on=["text_id","field_type","span_text","start_char","end_char"],
    how="left",
    indicator=True
)

In [101]:
merged["correct"] = merged["_merge"] == "both"
errors = merged[merged["_merge"] == "left_only"]

precision_table = (
    merged
    .groupby("field_type")
    .agg(
        correct_extractions=("correct","sum"),
        all_extractions=("correct","count")
    )
)

precision_table["precision"] = (
    precision_table["correct_extractions"] /
    precision_table["all_extractions"]
)

precision_table

,correct_extractions,all_extractions,precision
field_type,,,
AMOUNT,9,9,1.000000
DATE,79,80,0.987500
SCORE,5,14,0.357143


In [102]:
errors = errors[
    ["text_id","field_type","span_text","start_char","end_char","method"]
]

errors.head(10)

,text_id,field_type,span_text,start_char,end_char,method
16,13,SCORE,15:00,308,313,regex_score_v1
21,18,SCORE,2-3,714,717,regex_score_v1
25,20,SCORE,21-22,267,272,regex_score_v1
47,50,DATE,12 та 14 травня 2026,377,397,regex_date_v1
69,216,SCORE,0:3,930,933,regex_score_v1
70,216,SCORE,3:2,935,938,regex_score_v1
83,637,SCORE,13-15,1756,1761,regex_score_v1
93,648,SCORE,4-5,1072,1075,regex_score_v1
98,652,SCORE,13-15,813,818,regex_score_v1
102,658,SCORE,23-24,1038,1043,regex_score_v1


In [103]:
import pandas as pd

errors_table = pd.DataFrame([
["13","SCORE","15:00","regex_score_v1","визначено як рахунок","це час","не матчити HH:MM якщо є 'о','об'"],
["18","SCORE","2-3","regex_score_v1","визначено як рахунок","діапазон значень","фільтр слів 'види','години','до','між'"],
["20","SCORE","21-22","regex_score_v1","визначено як рахунок","діапазон днів","не матчити якщо є місяць"],
["50","DATE","12 та 14 травня 2026","regex_date_v1","визначено як дата","дві дати","заборонити патерн 'та'"],
["216","SCORE","0:3","regex_score_v1","визначено як рахунок","статистика","фільтр слова 'співвідношення'"],
["216","SCORE","3:2","regex_score_v1","визначено як рахунок","співвідношення","фільтр контексту"],
["637","SCORE","13-15","regex_score_v1","визначено як рахунок","діапазон днів","не матчити якщо є місяць"],
["648","SCORE","4-5","regex_score_v1","визначено як рахунок","діапазон днів","не матчити якщо є місяць"],
["652","SCORE","13-15","regex_score_v1","визначено як рахунок","діапазон днів","не матчити якщо є місяць"],
["658","SCORE","23-24","regex_score_v1","визначено як рахунок","діапазон днів","не матчити якщо є місяць"]
], columns=[
"text_id","field_type","span_text","method",
"що спрацювало","чому це помилка","яке правило додано"
])

errors_table

,text_id,field_type,span_text,method,що спрацювало,чому це помилка,яке правило додано
0,13,SCORE,15:00,regex_score_v1,визначено як рахунок,це час,"не матчити HH:MM якщо є 'о','об'"
1,18,SCORE,2-3,regex_score_v1,визначено як рахунок,діапазон значень,"фільтр слів 'види','години','до','між'"
2,20,SCORE,21-22,regex_score_v1,визначено як рахунок,діапазон днів,не матчити якщо є місяць
3,50,DATE,12 та 14 травня 2026,regex_date_v1,визначено як дата,дві дати,заборонити патерн 'та'
4,216,SCORE,0:3,regex_score_v1,визначено як рахунок,статистика,фільтр слова 'співвідношення'
5,216,SCORE,3:2,regex_score_v1,визначено як рахунок,співвідношення,фільтр контексту
6,637,SCORE,13-15,regex_score_v1,визначено як рахунок,діапазон днів,не матчити якщо є місяць
7,648,SCORE,4-5,regex_score_v1,визначено як рахунок,діапазон днів,не матчити якщо є місяць
8,652,SCORE,13-15,regex_score_v1,визначено як рахунок,діапазон днів,не матчити якщо є місяць
9,658,SCORE,23-24,regex_score_v1,визначено як рахунок,діапазон днів,не матчити якщо є місяць


In [104]:
errors_table_short = errors_table[
    ["span_text", "що спрацювало","чому це помилка","яке правило додано"]
]

In [ ]:
report = f"""
# Audit Summary — Lab 4 (Information Extraction)

## Датасет
- Corpus size: ~800 texts
- Gold subset: ~100 texts

## Типи полів
- Дата (DATE)
- Сума грощей (AMOUNT)
- Рахунок у змаганнях (SCORE)

## Оцінка якості

{precision_table}

## False Positives Analysis

{errors_table_short}

## Додано покращення
- Контекстна фільтрація для SCORE
- Визначення часу (ГГ:ХХ)
- Перевірка на основі місяця для DATE

## Висновок
Вилучення на основі регулярних виразів досить добре працює для структурованих шаблонів, таких як дати та результати змагань,
але вимагає контекстної фільтрації, щоб уникнути помилкових результатів.
"""

with open("audit_summary_lab4.md", "w", encoding="utf-8") as f:
    f.write(report)